In [1]:
import json
from pathlib import Path

import pandas as pd

sample = pd.read_csv("../data/sample_25.csv")
raw = json.loads(Path("../results/raw_claude-haiku-5-5.json").read_text())
master = pd.read_csv("../data/raw/CUAD_v1/master_clauses.csv")
cap = master.set_index("Filename")["Cap On Liability-Answer"]

false_alarms = [
    i for i, r in enumerate(raw)
    if (r["result"].get("uncapped_liability") or {}).get("present")
    and sample.loc[i, "Uncapped Liability-Answer"] != "Yes"
]

with_cap = sum(cap.get(sample.loc[i, "Filename"]) == "Yes" for i in false_alarms)
print(f"{len(false_alarms)} false alarms | in {with_cap} of them, the lawyers found a liability CAP\n")

for i in false_alarms[:3]:
    entry = raw[i]["result"]["uncapped_liability"]
    print("===", sample.loc[i, "Filename"][:70])
    print("Lawyers found a cap:", cap.get(sample.loc[i, "Filename"]))
    print("Claude's explanation:", entry.get("explanation"))
    for q in (entry.get("quotes") or [])[:2]:
        print("  QUOTE:", q[:400])
    print()


13 false alarms | in 8 of them, the lawyers found a liability CAP

=== EbixInc_20010515_10-Q_EX-10.3_4049767_EX-10.3_Co-Branding Agreement.pd
Lawyers found a cap: Yes
Claude's explanation: The indemnification obligations in Sections 11.1 and 11.2 for breach of warranties, representations, or covenants carry no monetary cap, and Section 14.1 only excludes indirect and consequential damages.
  QUOTE: eBix will defend, indemnify and hold About, its officers, directors, agents, employees and consultants (collectively the "About Indemnified Parties") harmless from and against any third party claims against About for all liabilities, claims, actions, damages, losses, settlements, and costs

=== PapaJohnsInternationalInc_20190617_8-K_EX-10.1_11707365_EX-10.1_Endors
Lawyers found a cap: Yes
Claude's explanation: The liability cap in Section 8.G protects only ABG, Authentic Brands Group and CELEBRITY, while PAPA JOHN'S indemnification obligations in Section 8.B carry no cap.
  QUOTE: PAPA JOHN'